# DESI Ib TF Secondary Target Selection

## sga_semiminor_ends
### Method to create the secondary targeting file for the mid-points on the semi-minor axis of large galaxies (from the SGA) in the BGS sample

##### Author: Kelly Douglass (University of Rochester)

See `/project/projectdirs/desi/target/secondary/README` for output data model

### Target classes
1. Mid-points on the major axis
2. **Mid-points on the minor axis**
3. Points along the major axis
4. Points off-axis

In [1]:
from astropy.table import Table, Column
from astropy.io import fits
from astropy import units as u
from astropy.coordinates import SkyCoord

import numpy as np

## Parameters

In [2]:
OVERRIDE = True
REF_EPOCH = 2015.5

#output_directory = '/project/projectdirs/desi/target/secondary/indata/'
output_directory = ''

## Target catalogs

[Siena Galaxy Atlas](https://www.legacysurvey.org/sga/sga2020/)

In [3]:
# Target catalog file names

#input_directory = '/Users/kellydouglass/Documents/Research/data/SGA/'
input_directory = '/global/cfs/cdirs/cosmo/work/legacysurvey/sga/2025/'

input_filename = input_directory + 'SGA2025-beta-parent-refcat-v1.6.kd.fits'

hdul = fits.open(input_filename)
large_galaxies = Table(hdul[1].data)
hdul.close()

## Set third priority: points in the middle of the minor axis

## Calculate (ra, dec) of minor axis mid-points

We set two fiber target locations on the minor axis at a distance of $0.5R(26)$.

In [4]:
center_ra = large_galaxies['ra']   # degrees
center_dec = large_galaxies['dec'] # degrees

centers = SkyCoord(center_ra*u.deg, center_dec*u.deg)

phi = large_galaxies['pa']*u.deg
r26 = 0.5*large_galaxies['diam']*u.arcmin
ba = large_galaxies['ba']

x = 0.5

# Maximum distance along the semi-minor axis from the center coordinate for our endpoints
delta_b = x*r26*ba

# Target positions
fiber1 = centers.directional_offset_by(phi + 90*u.deg, delta_b)
fiber2 = centers.directional_offset_by(phi - 90*u.deg, delta_b)

fiber_ra = np.concatenate((fiber1.ra, fiber2.ra))
fiber_dec = np.concatenate((fiber1.dec, fiber2.dec))

### Write target list to file

In [5]:
N_targets = len(fiber_ra)

In [6]:
disk_edges = Table([Column(fiber_ra, name='RA'), 
                    Column(fiber_dec, name='DEC'), 
                    Column(np.zeros(N_targets, dtype='>f4'), name='PMRA'), 
                    Column(np.zeros(N_targets, dtype='>f4'), name='PMDEC'), 
                    Column(REF_EPOCH*np.ones(N_targets, dtype='>f4'), name='REF_EPOCH'),
                    Column(OVERRIDE*np.ones(N_targets, dtype='bool'), name='OVERRIDE')])

In [7]:
disk_edges.write(output_directory + 'sga2025_semiminor_ends.fits', format='fits')

## Target statistics

In [8]:
print('The number of SGA-2025 galaxies is', len(large_galaxies))
print('The number of targets is', N_targets)

sky_chunk_boolean = np.logical_and.reduce([disk_edges['RA'] > 150, disk_edges['RA'] < 250, 
                                           disk_edges['DEC'] > 0, disk_edges['DEC'] < 50])

num_targets_in_sky_chunk = np.sum(sky_chunk_boolean)

sky_area = 100*50

print('The number of SGA galaxies in this portion of the sky is', 
      0.5*num_targets_in_sky_chunk)
print('The number of SGA galaxies per square degree is', 
      0.5*num_targets_in_sky_chunk/sky_area)
print('The number of fiber placements in this portion of the sky is', 
      num_targets_in_sky_chunk)
print('The number of fiber placements per square degree is', 
      num_targets_in_sky_chunk/sky_area)

The number of SGA-2025 galaxies is 470625
The number of targets is 941250
The number of SGA galaxies in this portion of the sky is 93335.0
The number of SGA galaxies per square degree is 18.667
The number of fiber placements in this portion of the sky is 186670
The number of fiber placements per square degree is 37.334
